# Notebook 07 — Deployment End-to-End Validation

Kiểm thử thật chuỗi **RAW INPUT → FEATURE ENGINEERING → MODEL_FEATURES → MODEL → PREDICTION**, đồng thời kiểm tra cluster/recommend endpoints và bốn tab Streamlit.

In [1]:
from pathlib import Path
import json
import sys

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    for candidate in Path.cwd().resolve().parents:
        if (candidate / "src").exists() and (candidate / "5.DATA").exists():
            ROOT = candidate
            break
sys.path.insert(0, str(ROOT))
print(f"Project root: {ROOT}")

import importlib.util
from fastapi.testclient import TestClient
from streamlit.testing.v1 import AppTest

from src.features import RAW_INPUT_FEATURES, get_model_features

MODEL_DIR = ROOT / "4.MODELS" / "hitradar_popularity"
SECONDARY_DIR = ROOT / "4.MODELS" / "hitradar_secondary"
DATA_PATH = ROOT / "5.DATA" / "processed" / "ml_ready_dataset.parquet"
metrics = json.loads((MODEL_DIR / "metrics.json").read_text(encoding="utf-8"))
pipeline = joblib.load(MODEL_DIR / "popularity_pipeline.joblib")
expected_features = get_model_features(include_engineered=metrics["include_engineered"], include_time=metrics["include_time"])
assert metrics["model_features"] == expected_features
print("Actual final experiment:", metrics["final_experiment"], "/", metrics["final_model"])
print("Model feature count:", len(expected_features))

Project root: D:\Hitradar\hitradar-main


D:\Hitradar\hitradar-main\scratch\runtime_packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


Actual final experiment: Baseline With-Time / XGBoost
Model feature count: 18


In [2]:
raw_data = pd.read_parquet(DATA_PATH)
raw_example = raw_data.loc[[raw_data.index[-1]], RAW_INPUT_FEATURES]
engineered = pipeline.named_steps["features"].transform(raw_example)
model_features_present = all(f in engineered.columns for f in expected_features)
prediction_raw = float(pipeline.predict(raw_example)[0])
prediction = float(np.clip(prediction_raw, 0, 100))
e2e = {"raw_columns_ok":raw_example.columns.tolist()==RAW_INPUT_FEATURES,
       "feature_engineering_ok":model_features_present,
       "model_features":len(expected_features), "prediction_raw":prediction_raw,
       "prediction_clipped":prediction, "status":"PASS"}
assert np.isfinite(prediction)
display(pd.DataFrame([e2e]))

,raw_columns_ok,feature_engineering_ok,model_features,prediction_raw,prediction_clipped,status
0,True,True,18,37.203003,37.203003,PASS


## FastAPI integration: prediction, cluster, recommendation

In [3]:
api_path = ROOT / "5.UNG_DUNG" / "5.1.backend_api" / "api.py"
spec = importlib.util.spec_from_file_location("hitradar_hotfix_api", api_path)
api_module = importlib.util.module_from_spec(spec); spec.loader.exec_module(api_module)
client = TestClient(api_module.app)
payload = raw_example.iloc[0].to_dict()
payload["explicit"] = bool(payload["explicit"])
for key in ("release_year","key","mode","time_signature","release_month"):
    payload[key] = int(payload[key])

health = client.get("/health")
pred_response = client.post("/predict", json=payload)
cluster_response = client.post("/cluster", json=payload)
query_id = str(raw_data.iloc[0]["track_id"])
recommend_response = client.get(f"/recommend/{query_id}?n=5")
assert health.status_code == pred_response.status_code == cluster_response.status_code == recommend_response.status_code == 200
assert query_id not in {r["track_id"] for r in recommend_response.json()["recommendations"]}
display(pd.DataFrame([health.json()]))
display(pd.DataFrame([pred_response.json()]))
display(pd.DataFrame([cluster_response.json()]))
display(pd.DataFrame(recommend_response.json()["recommendations"]))

,status,model_loaded,model_name,final_experiment,raw_input_count,model_feature_count,cluster_ready,recommender_ready
0,ready,True,XGBRegressor,Baseline With-Time,17,18,True,True


,predicted_popularity,popularity_tier,model_name,engineered_feature_count,feature_count
0,37.203,emerging,XGBoost,0,18


,cluster,chosen_k,feature_count
0,0,3,10


,track_id,cosine_similarity
0,2YSuy9tg3xtb2sKhvMRT3b,0.950803
1,5Nn2Dj7OQsGL6pgQ9iIzPp,0.947264
2,06DXs2hBRdjNs1qE1iYCQQ,0.946462
3,7vGxMdSqL9dSZsoSnOnLL6,0.940326
4,6BWRvw630R8z2vNMok6quI,0.934900


## Streamlit integration: bốn tab

In [4]:
streamlit_path = ROOT / "5.UNG_DUNG" / "5.2.frontend" / "streamlit_app.py"
app_test = AppTest.from_file(str(streamlit_path)).run(timeout=40)
tab_labels = [tab.label for tab in app_test.tabs]
streamlit_result = {"exceptions":len(app_test.exception), "tabs":tab_labels,
                    "status":"PASS" if not app_test.exception and len(tab_labels)==4 else "FAIL"}
assert streamlit_result["status"] == "PASS", streamlit_result
display(pd.DataFrame([streamlit_result]))

2026-08-13 21:54:15.868 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


2026-08-13 21:54:16.115 WARNING streamlit.config: 
'server.enableXsrfProtection=true'.
As a result, 'server.enableCORS' is being overridden to 'true'.

More information:
In order to protect against CSRF attacks, we send a cookie with each request.
To do so, we must specify allowable origins, which places a restriction on
cross-origin resource sharing.

If cross origin resource sharing is required, please disable server.enableXsrfProtection.
            


2026-08-13 21:54:16.119 DEBUG   streamlit.runtime.scriptrunner.script_runner: Running script


2026-08-13 21:54:16.120 DEBUG   streamlit.runtime.media_file_manager: Disconnecting files for session with ID test session id


2026-08-13 21:54:16.120 DEBUG   streamlit.runtime.media_file_manager: Sessions still active: dict_keys([])


2026-08-13 21:54:16.121 DEBUG   streamlit.runtime.media_file_manager: Files: 0; Sessions with files: 0


2026-08-13 21:54:16.611 DEBUG   streamlit.runtime.media_file_manager: Removing orphaned files...


2026-08-13 21:54:16.612 DEBUG   streamlit.runtime.media_file_manager: Removing orphaned deferred callables...


,exceptions,tabs,status
0,0,"[Overview, Popularity Prediction, Song Cluster...",PASS


In [5]:
validation = {"pipeline":e2e, "health":health.json(), "prediction":pred_response.json(),
              "cluster":cluster_response.json(), "recommendation":recommend_response.json(),
              "streamlit":streamlit_result}
validation_path = ROOT / "5.UNG_DUNG" / "validation" / "hotfix_end_to_end_validation.json"
validation_path.parent.mkdir(parents=True, exist_ok=True)
validation_path.write_text(json.dumps(validation, indent=2, ensure_ascii=False), encoding="utf-8")
print("Saved:", validation_path)
print("HOTFIX END-TO-END STATUS: PASS")

Saved: D:\Hitradar\hitradar-main\5.UNG_DUNG\validation\hotfix_end_to_end_validation.json
HOTFIX END-TO-END STATUS: PASS


## Kết luận

Deployment tải đúng winner thật từ Notebook 06, nhận raw inputs, tái tạo features trong pipeline, clip popularity về [0,100], và phục vụ cluster/recommendation từ artifacts Notebook 05. Streamlit có đúng bốn tab: Overview, Popularity Prediction, Song Clustering, Similar Songs.